In [ ]:
# Cell 1 — Verify GPU
import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU — go back and set T4 GPU")

GPU Available: True
GPU Name: Tesla T4


In [ ]:
# Cell 2 — Confirm libraries loaded (run again after restart)
import transformers
import datasets
import nltk
import spacy
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)
print("All libraries loaded successfully")

Transformers version: 5.0.0
Datasets version: 4.0.0
All libraries loaded successfully


In [ ]:
# Cell 3 — Download CNN/DailyMail Dataset
from datasets import load_dataset

print("Downloading CNN/DailyMail dataset... this will take 3-5 minutes")

dataset = load_dataset("cnn_dailymail", "3.0.0")

print("\nDataset downloaded successfully")
print("Training samples:", len(dataset["train"]))
print("Validation samples:", len(dataset["validation"]))
print("Test samples:", len(dataset["test"]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]


Dataset downloaded successfully
Training samples: 287113
Validation samples: 13368
Test samples: 11490


In [ ]:
# Cell 4 — Explore raw data structure
print("=== DATASET STRUCTURE ===")
print(dataset)

print("\n=== SAMPLE ARTICLE (first 500 characters) ===")
print(dataset["train"][0]["article"][:500])

print("\n=== SAMPLE HIGHLIGHT/SUMMARY ===")
print(dataset["train"][0]["highlights"])

print("\n=== AVERAGE ARTICLE LENGTH ===")
sample = dataset["train"].select(range(1000))
avg_length = sum(len(x["article"].split()) for x in sample) / 1000
print(f"Average words per article (sample of 1000): {avg_length:.0f} words")

=== DATASET STRUCTURE ===
DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

=== SAMPLE ARTICLE (first 500 characters) ===
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s

=== SAMPLE HIGHLIGHT/SUMMARY ===
Harry Potter star Daniel Radcliffe gets £20M fortune as he tu

In [4]:
# Cell 5 — Text Preprocessing
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize
import re
import string

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load spacy model for lemmatization
nlp = spacy.load('en_core_web_sm')

# Initialize tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    """
    Full preprocessing pipeline:
    1. Lowercase
    2. Remove special characters
    3. Tokenize
    4. Remove stop words
    5. Stemming
    6. Lemmatization
    """
    # Step 1 — Lowercase
    text = text.lower()

    # Step 2 — Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 3 — Tokenize
    tokens = word_tokenize(text)

    # Step 4 — Remove stop words
    tokens_no_stopwords = [t for t in tokens if t not in stop_words]

    # Step 5 — Stemming
    tokens_stemmed = [stemmer.stem(t) for t in tokens_no_stopwords]

    # Step 6 — Lemmatization using spacy
    doc = nlp(' '.join(tokens_no_stopwords))
    tokens_lemmatized = [token.lemma_ for token in doc]

    return {
        'original': text[:200],
        'after_stopword_removal': ' '.join(tokens_no_stopwords[:20]),
        'after_stemming': ' '.join(tokens_stemmed[:20]),
        'after_lemmatization': ' '.join(tokens_lemmatized[:20])
    }

# Test on first article
sample_article = dataset["train"][0]["article"]
result = preprocess_text(sample_article)

print("=== PREPROCESSING PIPELINE DEMONSTRATION ===\n")
print("ORIGINAL TEXT (first 200 chars):")
print(result['original'])
print("\nAFTER STOP WORD REMOVAL (first 20 tokens):")
print(result['after_stopword_removal'])
print("\nAFTER STEMMING (first 20 tokens):")
print(result['after_stemming'])
print("\nAFTER LEMMATIZATION (first 20 tokens):")
print(result['after_lemmatization'])
print("\nPreprocessing pipeline working successfully")

=== PREPROCESSING PIPELINE DEMONSTRATION ===

ORIGINAL TEXT (first 200 chars):
london england reuters harry potter star daniel radcliffe gains access to a reported million million fortune as he turns on monday but he insists the money wont cast a spell on him daniel radcliffe as

AFTER STOP WORD REMOVAL (first 20 tokens):
london england reuters harry potter star daniel radcliffe gains access reported million million fortune turns monday insists money wont cast

AFTER STEMMING (first 20 tokens):
london england reuter harri potter star daniel radcliff gain access report million million fortun turn monday insist money wont cast

AFTER LEMMATIZATION (first 20 tokens):
london england reuters harry potter star daniel radcliffe gain access report million million fortune turn monday insist money will not

Preprocessing pipeline working successfully


In [ ]:
# Cell 6 — Preprocess and save cleaned dataset
import pandas as pd

print("Preprocessing 5000 articles from training set...")
print("(Using 5000 for manageability — full dataset is 287,113)")

# Select 5000 samples
sample_data = dataset["train"].select(range(5000))

cleaned_data = []
for i, item in enumerate(sample_data):
    if i % 500 == 0:
        print(f"  Processed {i}/5000 articles...")

    article = item["article"]
    highlights = item["highlights"]

    # Basic cleaning only for model use
    # (full preprocessing shown above is for EDA analysis)
    article_clean = re.sub(r'\s+', ' ', article).strip()
    highlights_clean = re.sub(r'\s+', ' ', highlights).strip()

    cleaned_data.append({
        'article': article_clean,
        'highlights': highlights_clean,
        'article_word_count': len(article_clean.split()),
        'highlights_word_count': len(highlights_clean.split())
    })

# Save to CSV
df = pd.DataFrame(cleaned_data)
df.to_csv('docsnap_cleaned_data.csv', index=False)

print(f"\nDataset saved successfully")
print(f"Total articles processed: {len(df)}")
print(f"Average article length: {df['article_word_count'].mean():.0f} words")
print(f"Average summary length: {df['highlights_word_count'].mean():.0f} words")
print(f"File saved as: docsnap_cleaned_data.csv")

Preprocessing 5000 articles from training set...
(Using 5000 for manageability — full dataset is 287,113)
  Processed 0/5000 articles...
  Processed 500/5000 articles...
  Processed 1000/5000 articles...
  Processed 1500/5000 articles...
  Processed 2000/5000 articles...
  Processed 2500/5000 articles...
  Processed 3000/5000 articles...
  Processed 3500/5000 articles...
  Processed 4000/5000 articles...
  Processed 4500/5000 articles...

Dataset saved successfully
Total articles processed: 5000
Average article length: 615 words
Average summary length: 44 words
File saved as: docsnap_cleaned_data.csv


In [ ]:
# Cell 7 — Verify saved data
import pandas as pd

df = pd.read_csv('docsnap_cleaned_data.csv')

print("=== SAVED DATASET VERIFICATION ===")
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nSample row:")
print(f"Article (first 150 chars): {df['article'][0][:150]}")
print(f"Summary: {df['highlights'][0]}")
print(f"\nWord count stats:")
print(df[['article_word_count', 'highlights_word_count']].describe().round(2))
print("\nData verified and ready for EDA tomorrow")

=== SAVED DATASET VERIFICATION ===
Total rows: 5000
Columns: ['article', 'highlights', 'article_word_count', 'highlights_word_count']

Sample row:
Article (first 150 chars): LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monda
Summary: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday . Young actor says he has no plans to fritter his cash away . Radcliffe's earnings from first five Potter films have been held in trust fund .

Word count stats:
       article_word_count  highlights_word_count
count             5000.00                5000.00
mean               615.21                  43.89
std                302.23                   7.73
min                 18.00                  11.00
25%                376.00                  38.00
50%                570.00                  45.00
75%                816.25                  50.00
max               1831.00            

In [ ]:
# Cell 8 — Download Research Papers Dataset (ArXiv)
from datasets import load_dataset

print("Downloading ArXiv research papers dataset...")

arxiv_data = load_dataset("ccdv/arxiv-summarization", split="train[:2000]")

print("\nResearch papers dataset downloaded")
print(f"Total papers: {len(arxiv_data)}")
print(f"\nSample paper title/abstract (first 300 chars):")
print(arxiv_data[0]['article'][:300])
print(f"\nSample summary:")
print(arxiv_data[0]['abstract'][:200])

README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00003-of-00015.parquet:   0%|          | 0.00/227M [00:00<?, ?B/s]

...

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]


Research papers dataset downloaded
Total papers: 2000

Sample paper title/abstract (first 300 chars):
additive models @xcite provide an important family of models for semiparametric regression or classification . some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fu

Sample summary:
additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for additive models . 
 these learning rates compare favour


In [ ]:
# Cell 9 — Save ArXiv data
import pandas as pd

arxiv_df = pd.DataFrame({
    'article': [x['article'] for x in arxiv_data],
    'summary': [x['abstract'] for x in arxiv_data]
})

arxiv_df.to_csv('docsnap_arxiv_data.csv', index=False)
print(f"ArXiv dataset saved")
print(f"Total papers: {len(arxiv_df)}")
print(f"File: docsnap_arxiv_data.csv")

ArXiv dataset saved
Total papers: 2000
File: docsnap_arxiv_data.csv


In [ ]:
# Cell 10 — Apply full preprocessing pipeline to all 5000 articles
import pandas as pd
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import re

# Reload everything after restart
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

nlp = spacy.load('en_core_web_sm')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Load saved data
df = pd.read_csv('docsnap_cleaned_data.csv')
print(f"Loaded {len(df)} articles")
print("Applying full preprocessing pipeline...")
print("(This will take 10-15 minutes)\n")

def full_preprocess(text):
    # Step 1 — Lowercase
    text = text.lower()
    # Step 2 — Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Step 3 — Tokenize
    tokens = word_tokenize(text)
    # Step 4 — Remove stop words
    tokens = [t for t in tokens if t not in stop_words]
    # Step 5 — Stemming
    tokens_stemmed = [stemmer.stem(t) for t in tokens]
    # Step 6 — Lemmatization
    doc = nlp(' '.join(tokens[:100]))  # limit for speed
    tokens_lemmatized = [token.lemma_ for token in doc]

    return {
        'preprocessed_tokens': ' '.join(tokens),
        'stemmed_tokens': ' '.join(tokens_stemmed),
        'lemmatized_tokens': ' '.join(tokens_lemmatized)
    }

# Apply to all articles
preprocessed_articles = []
stemmed_articles = []
lemmatized_articles = []

for i, row in df.iterrows():
    if i % 500 == 0:
        print(f"  Processing article {i}/5000...")

    result = full_preprocess(row['article'])
    preprocessed_articles.append(result['preprocessed_tokens'])
    stemmed_articles.append(result['stemmed_tokens'])
    lemmatized_articles.append(result['lemmatized_tokens'])

# Add to dataframe
df['preprocessed'] = preprocessed_articles
df['stemmed'] = stemmed_articles
df['lemmatized'] = lemmatized_articles

# Save
df.to_csv('docsnap_fully_preprocessed.csv', index=False)

print(f"\nFull preprocessing complete")
print(f"Total articles preprocessed: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nSample preprocessed article (first 100 tokens):")
print(df['preprocessed'][0][:200])
print(f"\nSample stemmed (first 100 tokens):")
print(df['stemmed'][0][:200])
print(f"\nSample lemmatized (first 100 tokens):")
print(df['lemmatized'][0][:200])
print(f"\nSaved as: docsnap_fully_preprocessed.csv")

Loaded 5000 articles
Applying full preprocessing pipeline...
(This will take 10-15 minutes)

  Processing article 0/5000...
  Processing article 500/5000...
  Processing article 1000/5000...
  Processing article 1500/5000...
  Processing article 2000/5000...
  Processing article 2500/5000...
  Processing article 3000/5000...
  Processing article 3500/5000...
  Processing article 4000/5000...
  Processing article 4500/5000...

Full preprocessing complete
Total articles preprocessed: 5000
Columns: ['article', 'highlights', 'article_word_count', 'highlights_word_count', 'preprocessed', 'stemmed', 'lemmatized']

Sample preprocessed article (first 100 tokens):
london england reuters harry potter star daniel radcliffe gains access reported million million fortune turns monday insists money wont cast spell daniel radcliffe harry potter harry potter order phoe

Sample stemmed (first 100 tokens):
london england reuter harri potter star daniel radcliff gain access report million million fortun t

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/DocSnap_Project', exist_ok=True)
print("Drive mounted")

Mounted at /content/drive
Drive mounted


In [ ]:
# Complete ArXiv Preprocessing + Save
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/DocSnap_Project', exist_ok=True)

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from datasets import load_dataset
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import spacy
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

print("Loading ArXiv dataset...")
arxiv_data = load_dataset("ccdv/arxiv-summarization", split="train[:2000]")
print(f"Loaded: {len(arxiv_data)} papers")

nlp = spacy.load('en_core_web_sm')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_paper(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens_stemmed = [stemmer.stem(t) for t in tokens]
    doc = nlp(' '.join(tokens[:100]))
    tokens_lemmatized = [token.lemma_ for token in doc]
    return {
        'preprocessed': ' '.join(tokens),
        'stemmed': ' '.join(tokens_stemmed),
        'lemmatized': ' '.join(tokens_lemmatized)
    }

print("Preprocessing 2000 papers...")
papers = []
for i, item in enumerate(arxiv_data):
    if i % 400 == 0:
        print(f"  Processing {i}/2000...")
    result = preprocess_paper(item['article'][:2000])
    papers.append({
        'article': item['article'],
        'summary': item['abstract'],
        'preprocessed': result['preprocessed'],
        'stemmed': result['stemmed'],
        'lemmatized': result['lemmatized'],
        'article_word_count': len(item['article'].split()),
        'summary_word_count': len(item['abstract'].split())
    })

arxiv_df = pd.DataFrame(papers)
arxiv_df.to_csv(
    '/content/drive/MyDrive/DocSnap_Project/arxiv_preprocessed.csv',
    index=False
)

print(f"\nComplete — {len(arxiv_df)} papers preprocessed and saved")
print(f"Average paper length: {arxiv_df['article_word_count'].mean():.0f} words")
print(f"File saved to Google Drive successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading ArXiv dataset...


README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

...

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

Loaded: 2000 papers
Preprocessing 2000 papers...
  Processing 0/2000...
  Processing 400/2000...
  Processing 800/2000...
  Processing 1200/2000...
  Processing 1600/2000...

Complete — 2000 papers preprocessed and saved
Average paper length: 5894 words
File saved to Google Drive successfully
